# GGA Network Training - Step 5: SCF Solver Exploration

This notebook extends step 4 by exploring **SCF self-consistency** during
training and evaluation. Instead of the single one-shot density evaluation
used in step 4, step 5 trains each architecture under three solver
configurations and compares the effect on atomization energies and density
quality.

## Solver Configurations

| Label | Mode | Max Cycles | Description |
|-------|------|-----------|-------------|
| **oneshot** | ONESHOT | 0 | Single-pass density evaluation (step 4 baseline) |
| **fixed_j_3** | FIXED_J | 3 | 3 SCF cycles with frozen Coulomb matrix |
| **full_3** | FULL | 3 | 3 SCF cycles with full Fock rebuild each iteration |

## Training Matrix

**8 deep architectures x 3 loss approaches x 3 solver configs = 72 runs**

### Loss Approaches

| Approach | Energy Calculation | Density Matching | Description |
|----------|-------------------|------------------|-------------|
| **A** | Fixed-density | None | AE only on PBE density |
| **B** | Fixed-density | One-shot DM -> HF target | AE + DM correction learning |
| **C** | Fixed-density | One-shot grid rho -> HF target | AE + grid density correction |

### Network Architectures (8 deep variants)

| Architecture | Inputs | Dimension |
|--------------|--------|-----------|
| `deep`, `deep_attn` | $[\rho, \sigma]$ | 2 |
| `deep_cusp`, `deep_cusp_attn` | $[\rho, \sigma, f_{cusp}, \log Z]$ | 4 |
| `deep_dm`, `deep_dm_attn` | $[\rho, \sigma, f_{idem}, f_{entropy}, f_{offdiag}]$ | 5 |
| `deep_combined`, `deep_combined_attn` | $[\rho, \sigma, f_{idem}, f_{entropy}, f_{offdiag}, f_{cusp}, \log Z]$ | 7 |

**Total: 72 models** = 8 architectures x 3 training approaches x 3 solver configs


In [ ]:
import os
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
# JAX config: pin x64 dtype and CPU device *before* importing jnp or any
# library that may trigger JAX tracing. These must not change later in the
# notebook -- flipping jax_enable_x64 after traces are cached produces
# inconsistent dtypes.
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_device", jax.devices("cpu")[0])
import jax.numpy as jnp
import equinox as eqx

from pyscf import gto, dft, scf, cc

import xcquinox.alec as alec
import xcquinox.features
from xcquinox.alec.solver import SolverConfig, SolverBackend, SolverMode

# tqdm.auto picks tqdm.notebook.tqdm (ipywidgets) under JupyterLab and
# tqdm.std.tqdm in a plain script/terminal, so the same symbol gives a
# sensible progress bar in either context.
from tqdm.auto import tqdm


In [ ]:
BASIS = 'def2-svp'
CHECKPOINT_BASE = 'checkpoints_step5'
GRID_LEVEL = 1
PRETRAIN_ATOMS = (("H", 1), ("He", 0), ("O", 2), ("N", 3))
H2O_COORDS = "O 0.0000 0.0000 0.1173; H 0.0000 0.7572 -0.4692; H 0.0000 -0.7572 -0.4692"

# Flip to True to skip pretraining for any arch that already has both
# ``xnet.eqx`` and ``cnet.eqx`` at ``CHECKPOINT_BASE/pretrain/<arch>/``.
PRETRAIN_SKIP_IF_EXISTS = False

# Flip to True to skip the main training loop for any (arch, loss, solver)
# run that already has a ``model.eqx`` at
# ``CHECKPOINT_BASE/train/<arch>/<loss_name>/<solver_label>/``.
TRAIN_SKIP_IF_EXISTS = False

os.makedirs(CHECKPOINT_BASE, exist_ok=True)
print(f"CHECKPOINT_BASE={CHECKPOINT_BASE}  BASIS={BASIS}  GRID_LEVEL={GRID_LEVEL}")


In [ ]:
# Print the deep-only registered architectures from alec.ARCHITECTURES.
# Step 5 focuses on deep variants only (8 total).
# Fields printed: name, depth, nodes (hidden size), attention flag, descriptors.
_deep_names = [n for n in alec.ARCHITECTURES.keys() if n.startswith("deep")]
_header = f"{'arch_name':<22} {'depth':>6} {'nodes':>6} {'attention':>10}  descriptors"
print(_header)
print("-" * len(_header))
for _name in _deep_names:
    _cfg = alec.get_architecture(_name)
    _descs = ", ".join(s.name for s in _cfg.descriptors) or "-"
    print(f"{_name:<22} {_cfg.depth:>6} {_cfg.nodes:>6} {str(_cfg.attention):>10}  {_descs}")
print(f"\n{len(_deep_names)} deep architectures selected")


In [ ]:
ARCH_NAMES = [n for n in alec.ARCHITECTURES.keys() if n.startswith("deep")]

cmap = plt.get_cmap("tab10")
arch_colors = {name: cmap(i / max(1, len(ARCH_NAMES) - 1)) for i, name in enumerate(ARCH_NAMES)}

print(f"Selected {len(ARCH_NAMES)} architectures:")
for _n in ARCH_NAMES:
    print(f"  {_n}")


In [ ]:
SCF_CONFIGS = {
    "oneshot": SolverConfig(
        backend=SolverBackend.MANUAL,
        mode=SolverMode.ONESHOT,
    ),
    "fixed_j_3": SolverConfig(
        backend=SolverBackend.MANUAL,
        mode=SolverMode.FIXED_J,
        max_cycles=3,
        conv_tol=1e-6,
    ),
    "full_3": SolverConfig(
        backend=SolverBackend.MANUAL,
        mode=SolverMode.FULL,
        max_cycles=3,
        conv_tol=1e-6,
    ),
}

SOLVER_LABELS = list(SCF_CONFIGS.keys())

cmap_solver = plt.get_cmap("Set2")
solver_colors = {label: cmap_solver(i / max(1, len(SOLVER_LABELS) - 1)) for i, label in enumerate(SOLVER_LABELS)}

print(f"Solver configs ({len(SOLVER_LABELS)}):")
for _label in SOLVER_LABELS:
    _cfg = SCF_CONFIGS[_label]
    print(f"  {_label}: mode={_cfg.mode.value}, max_cycles={_cfg.max_cycles}")


## Section 3: Pretraining

Before the main training loop, each network (xnet / cnet) is **pretrained** on
atomic PBE enhancement factors so the weights start near a meaningful baseline
instead of a cold random initialisation. Starting from random weights causes the
main training loss to diverge; pretraining on known-good PBE targets avoids this.

### Pretrain atoms

Four atoms are used: **H** (spin=1), **He** (spin=0), **O** (spin=2), **N**
(spin=3). Their DFT grids cover a wide range of densities and gradient norms,
giving xnet / cnet a representative sample of the `(rho, sigma)` input space.

### Target: PBE enhancement factors

For each atom the PBE exchange and correlation enhancement factors are computed
via `pyscf`'s `eval_xc` with the exact libxc functional strings (`"PBE,"` /
`",PBE"` for GGA, `"LDA_X,"` / `",LDA_C_PW"` for the LDA baseline). The
network targets are `F_x - 1` and `F_c - 1` (shift by 1 so the loss near PBE
is near zero).

### Low-density cutoff and clipping

Grid points with `rho <= 1e-10` are dropped at write time — below this threshold
the density is numerically zero and the enhancement factor is undefined. The
targets are clipped to `[-5, 5]` to suppress outliers in the atomic core and
tail regions that would otherwise dominate the loss.


In [ ]:
# Pretrain data generation (inline pyscf) — matches step3b Cell 10.
rho_list, sigma_list, Fx_list, Fc_list = [], [], [], []
cusp_list, dm_list = [], []

# Compute gate: only compute extended features iff ARCH_NAMES contains
# architectures that actually declare the corresponding descriptor.
_arch_objs = [alec.get_architecture(n) for n in ARCH_NAMES]
need_cusp = any(s.name == "cusp" for a in _arch_objs for s in a.descriptors)
need_dm = any(s.name == "dm_statistics" for a in _arch_objs for s in a.descriptors)

for atom_symbol, spin in PRETRAIN_ATOMS:
    mol = gto.M(atom=f"{atom_symbol} 0 0 0", basis=BASIS, charge=0, spin=spin, verbose=0)
    mf = dft.UKS(mol) if spin else dft.RKS(mol)
    mf.xc = "pbe"
    mf.grids.level = GRID_LEVEL
    mf.kernel()

    ao = mf._numint.eval_ao(mol, mf.grids.coords, deriv=1)
    dm_ab = mf.make_rdm1()
    dm_total = dm_ab[0] + dm_ab[1] if dm_ab.ndim == 3 else dm_ab
    rho_gga = mf._numint.eval_rho(mol, ao, dm_total, xctype="GGA", hermi=True)

    rho = rho_gga[0]
    sigma = rho_gga[1]**2 + rho_gga[2]**2 + rho_gga[3]**2

    # PBE enhancement factors from libxc (pyscf functional strings, NOT xcquinox helpers)
    ex_pbe = mf._numint.eval_xc("PBE,", rho_gga, spin=0)[0]
    ec_pbe = mf._numint.eval_xc(",PBE", rho_gga, spin=0)[0]
    # LDA baselines on the 1-D total density
    ex_lda = mf._numint.eval_xc("LDA_X,", rho, spin=0)[0]
    ec_lda = mf._numint.eval_xc(",LDA_C_PW", rho, spin=0)[0]

    # np.where-based safe division (NOT a boolean mask — boolean masks drop points
    # step3b keeps; spec Rounds 8-10 regression guard)
    ex_lda_safe = np.where(np.abs(ex_lda) > 1e-12, ex_lda, 1e-12)
    ec_lda_safe = np.where(np.abs(ec_lda) > 1e-12, ec_lda, 1e-12)
    Fx_minus_1 = ex_pbe / ex_lda_safe - 1.0
    Fc_minus_1 = ec_pbe / ec_lda_safe - 1.0

    Fx_minus_1 = np.clip(Fx_minus_1, -5.0, 5.0)
    Fc_minus_1 = np.clip(Fc_minus_1, -5.0, 5.0)

    # Low-density mask at write time — threshold is 1e-10 (NOT 1e-6),
    # strictly > (NOT >=). Step3b uses the looser cutoff to keep the atomic tail.
    valid = rho > 1e-10
    rho_write = rho[valid]
    sigma_write = sigma[valid]
    Fx_write = Fx_minus_1[valid]
    Fc_write = Fc_minus_1[valid]

    rho_list.append(rho_write)
    sigma_list.append(sigma_write)
    Fx_list.append(Fx_write)
    Fc_list.append(Fc_write)

    if need_cusp:
        coords_v = mf.grids.coords[valid]
        cusp_feat = xcquinox.features.compute_cusp_descriptor(
            jnp.asarray(coords_v),
            jnp.asarray(mol.atom_coords()),
            jnp.asarray(mol.atom_charges()),
        )
        cusp_list.append(np.asarray(cusp_feat))

    if need_dm:
        S = mol.intor("int1e_ovlp")
        dm_feat_global = xcquinox.features.compute_dm_features_array(
            jnp.asarray(dm_total), jnp.asarray(S)
        )
        dm_feat_tiled = jnp.tile(dm_feat_global, (len(rho_write), 1))
        dm_list.append(np.asarray(dm_feat_tiled))

rho_all   = np.concatenate(rho_list)
sigma_all = np.concatenate(sigma_list)
Fx_all    = np.concatenate(Fx_list)
Fc_all    = np.concatenate(Fc_list)

save_kwargs = dict(rho_all=rho_all, sigma_all=sigma_all, Fx_all=Fx_all, Fc_all=Fc_all)
if cusp_list:
    save_kwargs["cusp_all"] = np.concatenate(cusp_list)
if dm_list:
    save_kwargs["dm_all"] = np.concatenate(dm_list)

os.makedirs(os.path.join(CHECKPOINT_BASE, "pretrain_data"), exist_ok=True)
np.savez(os.path.join(CHECKPOINT_BASE, "pretrain_data", "pretrain_data.npz"), **save_kwargs)
print(f"pretrain_data.npz written with keys: {sorted(save_kwargs.keys())}  total_points={len(rho_all)}")


In [ ]:
# Per-(arch, phase) tqdm bars keyed by (arch_name, phase_letter).
# The bar for a given phase is created on the first callback for that phase
# and closed when step == total. Scientific-notation postfix ``loss=...``
# keeps small values readable without losing precision.
_bars = {}

def _cb(info):
    key = (info['arch'], info['phase'])
    if key not in _bars:
        _bars[key] = tqdm(
            total=info['total'],
            desc=f"{info['arch']:<20} {info['phase']}net",
            leave=True,
            dynamic_ncols=True,
        )
    bar = _bars[key]
    delta = info['step'] - bar.n
    if delta > 0:
        bar.update(delta)
    bar.set_postfix(loss=f"{info['loss']:.4e}")
    if info['step'] >= info['total']:
        bar.close()
        del _bars[key]

def _pretrain_checkpoints_exist(arch_name):
    import os as _os
    _ckdir = f"{CHECKPOINT_BASE}/pretrain/{arch_name}"
    return (
        _os.path.isfile(f"{_ckdir}/xnet.eqx")
        and _os.path.isfile(f"{_ckdir}/cnet.eqx")
    )

for arch_name in ARCH_NAMES:
    if PRETRAIN_SKIP_IF_EXISTS and _pretrain_checkpoints_exist(arch_name):
        print(f"[{arch_name}] cached xnet.eqx + cnet.eqx found — skipping pretrain")
        continue
    spec = alec.PretrainSpec(
        arch=alec.get_architecture(arch_name),
        data_dir=f"{CHECKPOINT_BASE}/pretrain_data",
        checkpoint_dir=f"{CHECKPOINT_BASE}/pretrain/{arch_name}",
        n_steps=1000,
        lr_start=1e-2,
        lr_end=1e-5,
        lr_decay_start=0.2,
        grad_clip=1.0,
    )
    alec.run_pretrain(spec, progress_callback=_cb)


In [ ]:
fig, (ax_x, ax_c) = plt.subplots(1, 2, figsize=(12, 4.5))
for arch_name in ARCH_NAMES:
    losses_x = np.load(f"{CHECKPOINT_BASE}/pretrain/{arch_name}/losses_x.npy")
    losses_c = np.load(f"{CHECKPOINT_BASE}/pretrain/{arch_name}/losses_c.npy")
    ax_x.semilogy(losses_x, color=arch_colors[arch_name], label=arch_name)
    ax_c.semilogy(losses_c, color=arch_colors[arch_name], label=arch_name)

ax_x.set_title(r"xnet: target $F_x - 1$ (PBE exchange enhancement)")
ax_x.set_xlabel("optimizer step")
ax_x.set_ylabel("MSE loss (log scale)")
ax_x.grid(True, which="both", ls=":", alpha=0.4)
ax_c.set_title(r"cnet: target $F_c - 1$ (PBE correlation enhancement)")
ax_c.set_xlabel("optimizer step")
ax_c.set_ylabel("MSE loss (log scale)")
ax_c.grid(True, which="both", ls=":", alpha=0.4)
# Legend outside right on the right subplot only (avoids cluttering both)
ax_c.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize="small",
    title="architecture",
)

fig.suptitle(
    "Pretraining loss vs step -- one curve per architecture "
    "(atoms: H, He, O, N at def2-svp)",
    fontsize=12,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
fig.savefig(f"{CHECKPOINT_BASE}/figures/pretrain_losses.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Load pretrain data (same .npz Cell 8 wrote)
_data = np.load(f"{CHECKPOINT_BASE}/pretrain_data/pretrain_data.npz")
_rho = _data["rho_all"]
_sigma = _data["sigma_all"]
Fx_target = _data["Fx_all"]
Fc_target = _data["Fc_all"]

# Build per-architecture descriptor input inline. Column order MUST match
# the library's private _assemble_pretrain_descriptors helper:
#   [rho, sigma, dm_all columns (if use_dm), cusp_all[:, 0:2] (if use_cusp)]
# dm comes BEFORE cusp. The helper is private — we reproduce the logic here.
def _build_input_array(arch):
    cols = [_rho, _sigma]
    _use_dm = any(s.name == "dm_statistics" for s in arch.descriptors)
    _use_cusp = any(s.name == "cusp" for s in arch.descriptors)
    if _use_dm:
        _dm = _data["dm_all"]
        for _i in range(_dm.shape[1]):
            cols.append(_dm[:, _i])
    if _use_cusp:
        cols.append(_data["cusp_all"][:, 0])
        cols.append(_data["cusp_all"][:, 1])
    return jnp.stack([jnp.asarray(c) for c in cols], axis=1)

n_arch = len(ARCH_NAMES)
fig, axes = plt.subplots(n_arch, 2, figsize=(10, 3 * n_arch), squeeze=False)
for row, arch_name in enumerate(ARCH_NAMES):
    arch = alec.get_architecture(arch_name)
    skel_xnet, skel_cnet = alec.create_network_pair(arch)
    xnet = eqx.tree_deserialise_leaves(
        f"{CHECKPOINT_BASE}/pretrain/{arch_name}/xnet.eqx", skel_xnet
    )
    cnet = eqx.tree_deserialise_leaves(
        f"{CHECKPOINT_BASE}/pretrain/{arch_name}/cnet.eqx", skel_cnet
    )
    input_array = _build_input_array(arch)
    # xnet(p) / cnet(p) already return the full enhancement factor F
    # (networks.py: ``return 1 + lobterm.squeeze()``), so predictions MUST
    # NOT be shifted by +1.0 again. Adding +1.0 here would produce a parity
    # plot with the y-axis offset by +1 relative to the x-axis.
    Fx_pred = jax.vmap(lambda p: xnet(p))(input_array)
    Fc_pred = jax.vmap(lambda p: cnet(p))(input_array)

    ax_x = axes[row, 0]
    ax_c = axes[row, 1]
    # Plot in F space (add 1.0 to target to match the prediction)
    ax_x.scatter(np.asarray(Fx_target) + 1.0, np.asarray(Fx_pred), s=2,
                 c=[arch_colors[arch_name]])
    _lo_x = float(min(np.min(Fx_target) + 1.0, np.min(Fx_pred)))
    _hi_x = float(max(np.max(Fx_target) + 1.0, np.max(Fx_pred)))
    ax_x.plot([_lo_x, _hi_x], [_lo_x, _hi_x], "k--", lw=0.8, label="y = x")
    ax_x.set_title(rf"{arch_name} -- $F_x$ parity")
    ax_x.set_xlabel(r"$F_x$ target (PBE exchange enhancement)")
    ax_x.set_ylabel(r"$F_x$ predicted (xnet)")
    ax_x.grid(True, ls=":", alpha=0.4)

    ax_c.scatter(np.asarray(Fc_target) + 1.0, np.asarray(Fc_pred), s=2,
                 c=[arch_colors[arch_name]])
    _lo_c = float(min(np.min(Fc_target) + 1.0, np.min(Fc_pred)))
    _hi_c = float(max(np.max(Fc_target) + 1.0, np.max(Fc_pred)))
    ax_c.plot([_lo_c, _hi_c], [_lo_c, _hi_c], "k--", lw=0.8, label="y = x")
    ax_c.set_title(rf"{arch_name} -- $F_c$ parity")
    ax_c.set_xlabel(r"$F_c$ target (PBE correlation enhancement)")
    ax_c.set_ylabel(r"$F_c$ predicted (cnet)")
    ax_c.grid(True, ls=":", alpha=0.4)

fig.suptitle(
    "Pretrain parity: per-architecture prediction vs PBE enhancement target "
    "(points on y=x are perfectly matched)",
    fontsize=12,
)
fig.tight_layout(rect=(0, 0, 1, 0.985))
os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
fig.savefig(f"{CHECKPOINT_BASE}/figures/pretrain_parity.png", dpi=150, bbox_inches="tight")
plt.show()


## Section 3: Training Data

The training molecules are the same as step4: **H** (atom, spin=1), **O**
(atom, spin=2), and **H2O** (molecule, spin=0) at the def2-svp basis.

### Reference Generation

Three levels of reference data are computed for each species:

- **PBE** total energies — used as the atom-energy anchors (`atom_energies` dict)
  so the NN's required XC correction stays on the order of single kcal/mol.
- **HF** density matrix and grid density — stored as density targets for H2O
  (`dm_target`, `rho_ref_grid`). Atoms skip density targets because degenerate
  HOMO eigenvalues make one-shot density numerically unstable.
- **CCSD** total energies — recorded in the sidecar JSON for post-training
  atomization-energy comparison.

### ERI Precompute

The **FULL** SCF mode rebuilds the Fock matrix at each cycle, which requires
the electron-repulsion integrals (ERI). Cell 16 calls
`precompute_fixed_density_data` with `required_keys=("eri",)` to ensure the
ERI tensor is cached in the molecule data dictionaries.


In [ ]:
# Literature atomic total energies in Hartree (negative, as they should be).
# Used ONLY by Cell 14 to write each atom's E_ref_literature sidecar value
# (TotalEnergyMetric compares against this scalar).
# H is exact: -0.5 Ha. O is literature total ~ -75.0673 Ha.
atom_energies_literature = {"H": -0.5, "O": -75.0673}

# targets dict: validator requires an entry for every molecule in TrainingSpec.molecules
# (config.py:523-525). Atom entries are never dereferenced at training time but must be
# finite floats — we set them to the literature atomic totals for consistency.
# The H2O entry is the POSITIVE-for-bound atomization energy in Hartree:
#   AE = E_atoms_sum - E_mol > 0 for a bound molecule
# Literature: AE(H2O) ~ 974.94 kJ/mol = 974.94 / 2625.5 Ha.
targets = {"H": -0.5, "O": -75.0673, "H2O": 974.94 / 2625.5}

ext_data_dir = f"{CHECKPOINT_BASE}/external_data"
os.makedirs(ext_data_dir, exist_ok=True)
print(
    f"ext_data_dir={ext_data_dir}  "
    f"targets={list(targets.keys())}  "
    f"atom_energies_literature={list(atom_energies_literature.keys())}"
)
# NOTE: The runtime name `atom_energies` (consumed by the training loss and
# AtomizationEnergyMetric) is defined at the end of Cell 14 from the PBE
# atomic totals computed there. Do not reference `atom_energies` before Cell 14.


In [ ]:
# HF/CCSD reference computation and external_data .npz generation.
# H2O uses H2O_COORDS from Cell 3 (equilibrium geometry, NOT a distorted 90-degree box).
_mols = [
    ("H", "H 0 0 0", 1),
    ("O", "O 0 0 0", 2),
    ("H2O", H2O_COORDS, 0),
]

# Accumulates PBE atomic total energies (one entry per element symbol) so
# that at the end of this cell we can bind `atom_energies` to a
# PBE-consistent dict. Using PBE here rather than literature values keeps
# the NN's required XC correction on the order of single kcal/mol in the
# post-hoc fixed-density framework; literature anchors would demand a
# ~100 kcal/mol correction which the NN cannot produce on a frozen density.
# Concretely: PBE/6-31G** gives ~-0.500 Ha for H and ~-74.87 Ha for O,
# vs literature -0.5 / -75.0673 Ha. The ~0.2 Ha (~125 kcal/mol) O gap is
# exactly the correction the NN would otherwise have to conjure on a
# frozen density. Using PBE anchors makes this gap vanish for isolated
# atoms and leaves only the molecular correlation/exchange gap for the NN.
atom_energies_pbe = {}

for name, atom, spin in _mols:
    # Identical gto.M kwargs to what precompute_fixed_density_data uses internally.
    mol = gto.M(atom=atom, basis=BASIS, charge=0, spin=spin, verbose=0)

    # PBE SCF with grid pinned to GRID_LEVEL (must match Cell 15/16 precompute grid).
    mf = dft.UKS(mol) if mol.spin else dft.RKS(mol)
    mf.xc = "pbe"
    mf.grids.level = GRID_LEVEL
    mf.kernel()
    E_pbe_total = float(mf.e_tot)

    # HF SCF (spin-branched).
    mf_hf = scf.UHF(mol) if mol.spin else scf.RHF(mol)
    mf_hf.kernel()
    E_hf_total = float(mf_hf.e_tot)

    # CCSD (spin-branched). Runs for every molecule purely for sidecar documentation.
    mycc = cc.UCCSD(mf_hf) if mol.spin else cc.CCSD(mf_hf)
    mycc.kernel()
    E_ccsd_total = float(mf_hf.e_tot + mycc.e_corr)

    if name in ("H", "O"):
        # Atom branch: degenerate HOMO eigenvalues make one-shot density targets
        # numerically unstable. Write ONLY E_ref_literature for atoms.
        # E_ref_literature is the LITERATURE atomic total (TotalEnergyMetric
        # compares against this).
        np.savez(
            os.path.join(ext_data_dir, f"{name}.npz"),
            E_ref_literature=atom_energies_literature[name],
        )
        # Record the PBE total for this atom — consumed by the AE anchor dict
        # at the end of this cell.
        atom_energies_pbe[name] = E_pbe_total
    else:
        # H2O branch: write HF DM as density target (NOT CCSD DM — step3b uses HF).
        dm_hf = mf_hf.make_rdm1()
        dm_hf_total = dm_hf[0] + dm_hf[1] if dm_hf.ndim == 3 else dm_hf

        # Grid density from HF DM via einsum on the AO grid.
        coords = mf.grids.coords
        ao_grid = mf._numint.eval_ao(mol, coords, deriv=0)
        rho_hf = np.einsum("ij,gi,gj->g", dm_hf_total, ao_grid, ao_grid)

        # The three keys below are the ONLY keys _ALLOWED_EXTERNAL_KEYS accepts
        # (data.py:17-21). E_ref_literature is the HF total (not the CCSD total),
        # because TotalEnergyMetric.E_error_hartree gauges against this scalar and
        # the density-matching losses (B/C/D2/D3) optimize toward the HF density.
        np.savez(
            os.path.join(ext_data_dir, f"{name}.npz"),
            dm_target=dm_hf,
            rho_ref_grid=rho_hf,
            ref_density_method="hf",
            E_ref_literature=float(mf_hf.e_tot),
        )

    # Sidecar JSON for every species — library .npz cannot carry extra keys,
    # so HF/CCSD/literature/PBE totals live here. Cell 25 reads E_ccsd_total
    # from this file for the CCSD atomization-energy reference line.
    with open(os.path.join(ext_data_dir, f"{name}_metadata.json"), "w") as _f:
        json.dump(
            {
                "E_hf_total": E_hf_total,
                "E_ccsd_total": E_ccsd_total,
                "E_lit_Ha": atom_energies_literature.get(name, None),
                "E_pbe_total": E_pbe_total,
            },
            _f,
            indent=2,
        )

# Bind the runtime name `atom_energies` to the PBE-consistent dict. This is
# the dict that flows into TrainingSpec.atom_energies and
# TestSpec.atom_energies. After the losses.py fix, both the training
# loss and AtomizationEnergyMetric compute atomization energy as
# `sum(atom_energies[Z] * n_Z) - E_mol`, so the training loss and evaluation
# metric agree exactly on every compound.
atom_energies = dict(atom_energies_pbe)
print(f"Reference data written to {ext_data_dir}")
_ae_str = {k: round(v, 6) for k, v in atom_energies.items()}
print(f"atom_energies (PBE-consistent) = {_ae_str}")


In [ ]:
mol_specs = [
    alec.MoleculeSpec(
        name="H",
        atom="H 0 0 0",
        basis=BASIS,
        charge=0,
        spin=1,
        atom_composition=(("H", 1),),
        external_data_path=f"{ext_data_dir}/H.npz",
        grid_level=GRID_LEVEL,
    ),
    alec.MoleculeSpec(
        name="O",
        atom="O 0 0 0",
        basis=BASIS,
        charge=0,
        spin=2,
        atom_composition=(("O", 1),),
        external_data_path=f"{ext_data_dir}/O.npz",
        grid_level=GRID_LEVEL,
    ),
    alec.MoleculeSpec(
        name="H2O",
        atom=H2O_COORDS,
        basis=BASIS,
        charge=0,
        spin=0,
        atom_composition=(("H", 2), ("O", 1)),
        external_data_path=f"{ext_data_dir}/H2O.npz",
        grid_level=GRID_LEVEL,
    ),
]
print(f"Built {len(mol_specs)} MoleculeSpec objects: {[m.name for m in mol_specs]}")


In [ ]:
# Precompute mol_data for all molecules. required_keys includes "eri"
# because FULL SCF mode rebuilds the Fock matrix and needs the 4-center integrals.
_arch_objs = [alec.get_architecture(n) for n in ARCH_NAMES]
_desc_keys = set()
for _a in _arch_objs:
    for _d in _a.materialize_descriptors():
        _desc_keys.update(_d.required_mol_keys)

mol_data_list = []
for ms in mol_specs:
    md = alec.precompute_fixed_density_data(
        ms,
        required_keys=tuple(_desc_keys | {"eri"}),
        descriptors=sum((_a.materialize_descriptors() for _a in _arch_objs), ()),
    )
    mol_data_list.append(md)

# Sanity: atoms have 1-element composition, molecules have > 1
for md in mol_data_list:
    _n_atoms = sum(n for _, n in md["atom_composition"])
    _kind = "atom" if _n_atoms == 1 else "molecule"
    print(f"  {md['name']:5s}  ({_kind})  grid_pts={len(md['rho_grid'])}  keys={sorted(md.keys())[:8]}...")
    if "eri" in md:
        print(f"         ERI shape: {md['eri'].shape}")


## Section 4: SCF-Varied Training

This is the core experiment: **72 training runs** = 8 architectures x 3 loss
families x 3 solver configurations.

### Solver Impact by Loss Family

- **Loss A** (`A_atomization`): Energy-only. Uses `fixed_density_total_energy`
  which does NOT route through the SCF solver -- it always computes a one-shot
  energy on the PBE density. This is the **control experiment**: all 3 solver
  configs produce identical models for loss A.

- **Loss B** (`B_atomization_plus_dm`): Energy + density matrix term. The DM
  term (`_dm_term`) uses the solver to compute a self-consistent density matrix
  when solver_config is non-ONESHOT. With FIXED_J, the Coulomb operator is
  frozen during SCF cycles; with FULL, the full Fock matrix is rebuilt.

- **Loss C** (`C_atomization_plus_grid`): Energy + grid density term. Same SCF
  routing as loss B but applied to the grid-space density comparison.

### Expected Compute Cost

ONESHOT is fastest (single forward pass). FIXED_J(3 cycles) is ~3x slower.
FULL(3 cycles) is the most expensive due to ERI contraction at each cycle.


In [ ]:
LOSS_NAMES = (
    "A_atomization",
    "B_atomization_plus_dm",
    "C_atomization_plus_grid",
)

LOSS_KWARGS_BASE = {
    "A_atomization": {},
    "B_atomization_plus_dm": {"dm_weight": 0.1},
    "C_atomization_plus_grid": {"density_weight": 0.1},
}

specs = []
for arch_name in ARCH_NAMES:
    for loss_name in LOSS_NAMES:
        for solver_label in SOLVER_LABELS:
            cfg = SCF_CONFIGS[solver_label]
            # solver_config flows through loss_kwargs to make_loss -> loss ctor.
            # For loss A (energy-only), solver_config is accepted but ignored.
            _lkw = {**LOSS_KWARGS_BASE[loss_name], "solver_config": cfg}
            specs.append(alec.TrainingSpec.from_dicts(
                arch=alec.get_architecture(arch_name),
                loss_name=loss_name,
                molecules=tuple(mol_specs),
                targets=targets,
                atom_energies=atom_energies,
                loss_kwargs=_lkw,
                solver_config=cfg,
                pretrain_checkpoint=f"{CHECKPOINT_BASE}/pretrain/{arch_name}",
                checkpoint_dir=f"{CHECKPOINT_BASE}/train/{arch_name}/{loss_name}/{solver_label}",
                n_steps=250,
                lr_start=1e-2,
                lr_end=1e-5,
                lr_decay_start=0.2,
                grad_clip=1.0,
            ))
print(f"Built {len(specs)} training specs "
      f"({len(ARCH_NAMES)} archs x {len(LOSS_NAMES)} losses x {len(SOLVER_LABELS)} solvers)")


In [ ]:
_step_bars = {}
_current_info = {"loss": None, "solver": None}

def _train_cb(info):
    key = (info['arch'], info['phase'])
    if key not in _step_bars:
        _label = (f"{info['arch']:<20} {_current_info['loss']:<25} {_current_info['solver']}"
                  if _current_info['loss'] is not None
                  else f"{info['arch']:<20} {info['phase']}")
        _step_bars[key] = tqdm(
            total=info['total'],
            desc=_label,
            leave=False,
            dynamic_ncols=True,
        )
    bar = _step_bars[key]
    delta = info['step'] - bar.n
    if delta > 0:
        bar.update(delta)
    bar.set_postfix(loss=f"{info['loss']:.4e}")
    if info['step'] >= info['total']:
        bar.close()
        del _step_bars[key]

def _training_model_exists(spec):
    import os as _os
    return _os.path.isfile(_os.path.join(spec.checkpoint_dir, "model.eqx"))

_spec_bar = tqdm(
    total=len(specs),
    desc="training (specs)",
    leave=True,
    dynamic_ncols=True,
)
try:
    for spec in specs:
        _current_info['loss'] = spec.loss_name
        _current_info['solver'] = spec.checkpoint_dir.split('/')[-1]
        if TRAIN_SKIP_IF_EXISTS and _training_model_exists(spec):
            print(f"[{spec.arch.name}][{spec.loss_name}][{_current_info['solver']}] "
                  f"cached model.eqx found -- skipping training")
            _spec_bar.update(1)
            continue
        alec.run_training(spec, progress_callback=_train_cb)
        _spec_bar.update(1)
        _spec_bar.set_postfix(
            arch=spec.arch.name, loss=spec.loss_name,
            solver=_current_info['solver'])
finally:
    _spec_bar.close()
    for _b in list(_step_bars.values()):
        _b.close()
    _step_bars.clear()


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 13), squeeze=False)
for row_idx, solver_label in enumerate(SOLVER_LABELS):
    for col_idx, loss_name in enumerate(LOSS_NAMES):
        ax = axes[row_idx, col_idx]
        for arch_name in ARCH_NAMES:
            ckpt_dir = f"{CHECKPOINT_BASE}/train/{arch_name}/{loss_name}/{solver_label}"
            losses_path = f"{ckpt_dir}/losses.npy"
            if not os.path.isfile(losses_path):
                continue
            losses = np.load(losses_path)
            ax.semilogy(losses, color=arch_colors[arch_name], label=arch_name)
        ax.set_title(f"{solver_label} / {loss_name}", fontsize=10)
        ax.set_xlabel("training step")
        ax.set_ylabel("total loss (log)")
        ax.grid(True, which="both", ls=":", alpha=0.4)

# Shared legend from top-right subplot
axes[0, 2].legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize="small",
    title="architecture",
)

fig.suptitle(
    "Training loss curves -- rows: solver config, columns: loss family\n"
    "(8 deep architectures per subplot, one trace per arch)",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
fig.savefig(f"{CHECKPOINT_BASE}/figures/training_losses.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
arch_name = "deep_combined"
_aux_keys_per_family = {
    "A_atomization": ("loss_energy", "atomic_reg"),
    "B_atomization_plus_dm": ("loss_energy", "atomic_reg", "loss_dm"),
    "C_atomization_plus_grid": ("loss_energy", "atomic_reg", "loss_grid"),
}

fig, axes = plt.subplots(len(SOLVER_LABELS), len(LOSS_NAMES),
                         figsize=(15, 4 * len(SOLVER_LABELS)), squeeze=False)
for row_idx, solver_label in enumerate(SOLVER_LABELS):
    for col_idx, loss_name in enumerate(LOSS_NAMES):
        ax = axes[row_idx, col_idx]
        ckpt_dir = f"{CHECKPOINT_BASE}/train/{arch_name}/{loss_name}/{solver_label}"
        aux_path = f"{ckpt_dir}/aux_log.pkl"
        if not os.path.isfile(aux_path):
            ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center")
            ax.set_title(f"{solver_label} / {loss_name}", fontsize=10)
            continue
        with open(aux_path, "rb") as _f:
            aux_log = pickle.load(_f)

        _steps = [entry["step"] for entry in aux_log]
        for key in _aux_keys_per_family.get(loss_name, ("loss_energy",)):
            _vals = [entry["aux"].get(key, float("nan")) for entry in aux_log]
            ax.semilogy(_steps, _vals, label=key)
        ax.set_title(f"{solver_label} / {loss_name}", fontsize=10)
        ax.set_xlabel("training step")
        ax.set_ylabel("loss component (log scale)")
        ax.grid(True, which="both", ls=":", alpha=0.4)
        ax.legend(fontsize="small", loc="best")

fig.suptitle(
    f"Aux loss components for arch = {arch_name!r}\n"
    f"rows: solver config, columns: loss family",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
fig.savefig(f"{CHECKPOINT_BASE}/figures/aux_components_{arch_name}.png", dpi=150, bbox_inches="tight")
plt.show()


## Section 5: Evaluation

Each trained model is scored on the same molecules used for training. Four
metrics are computed per molecule:

- **`total_energy`** -- NN total energy vs PBE/HF reference.
- **`atomization_energy`** -- AE_nn vs literature (233.016 kcal/mol for H2O).
- **`density_rmse`** -- RMSE of grid density vs HF target (molecules only).
- **`constraint_violations`** -- flattened constraint report.

The evaluation loop sweeps all 72 (arch, loss, solver_config) combinations.
Each TestSpec carries the solver_config for metadata logging.


In [ ]:
for arch_name in ARCH_NAMES:
    for loss_name in LOSS_NAMES:
        for solver_label in SOLVER_LABELS:
            cfg = SCF_CONFIGS[solver_label]
            ckpt_dir = f"{CHECKPOINT_BASE}/train/{arch_name}/{loss_name}/{solver_label}"
            model_path = f"{ckpt_dir}/model.eqx"
            if not os.path.isfile(model_path):
                continue
            test_spec = alec.TestSpec.from_dicts(
                arch=alec.get_architecture(arch_name),
                model_checkpoint=model_path,
                molecules=tuple(mol_specs),
                metrics=("total_energy", "atomization_energy", "density_rmse", "constraint_violations"),
                metric_kwargs={"atomization_energy": {"reference_ae_kcalmol": {"H2O": 233.016}}},
                atom_energies=atom_energies,
                output_dir=f"{CHECKPOINT_BASE}/eval/{arch_name}/{loss_name}/{solver_label}",
                solver_config=cfg,
            )
            alec.run_test(test_spec)


In [ ]:
rows = []
for arch_name in ARCH_NAMES:
    for loss_name in LOSS_NAMES:
        for solver_label in SOLVER_LABELS:
            output_dir = f"{CHECKPOINT_BASE}/eval/{arch_name}/{loss_name}/{solver_label}"
            try:
                with open(f"{output_dir}/aggregate.json") as _f:
                    agg = json.load(_f)
            except FileNotFoundError:
                agg = {}
            rows.append({
                "arch": arch_name,
                "loss": loss_name,
                "solver": solver_label,
                "AE_error_kcalmol_mean": agg.get("AE_error_kcalmol", {}).get("mean", np.nan),
                "AE_error_kcalmol_RMSE": agg.get("AE_error_kcalmol", {}).get("RMSE", np.nan),
                "E_error_kcalmol_mean": agg.get("E_error_kcalmol", {}).get("mean", np.nan),
                "density_rmse_mean": agg.get("density_rmse", {}).get("mean", np.nan),
            })
df = pd.DataFrame(rows).set_index(["arch", "loss", "solver"])
print(f"Built results DataFrame: {df.shape[0]} rows x {df.shape[1]} cols")


In [ ]:
# Pivot: mean |AE error| per (arch, loss) x solver
piv = df["AE_error_kcalmol_mean"].abs().unstack(level="solver")
print("Mean |AE error| (kcal/mol), (arch, loss) x solver:")
print(piv.round(3))
print()
# Best config per loss
for loss_name in LOSS_NAMES:
    _sub = df.xs(loss_name, level="loss")["AE_error_kcalmol_mean"].abs()
    _best = _sub.idxmin()
    print(f"Best config for {loss_name}: arch={_best[0]}, solver={_best[1]}, "
          f"|AE err|={_sub[_best]:.3f} kcal/mol")


## Section 6: SCF Impact Analysis

This section compares trained models across the three solver configurations to
answer the central question: does training through an iterative SCF loop
produce better density functionals than one-shot prediction?

### Figure: SCF Comparison -- AE Error by Architecture

The next cell renders the **headline figure**: grouped bars showing |AE error|
for each architecture, with 3 bars per arch (oneshot / fixed_j / full). One
subplot per loss family (A, B, C). PBE error and 1 kcal/mol chemical accuracy
lines are overlaid as references.

**How to read it:** For loss A (energy only), all 3 solver configs should produce
nearly identical bars (control). For losses B and C, differences between solver
configs reveal the impact of SCF self-consistency on the learned functional.


In [ ]:
# Reference lines: PBE and CCSD atomization energy errors vs experiment
ext_data_dir = f"{CHECKPOINT_BASE}/external_data"
_E_ref = {}
for _name in ("H", "O", "H2O"):
    with open(f"{ext_data_dir}/{_name}_metadata.json") as _f:
        _E_ref[_name] = json.load(_f)
_AE_expt_kcalmol = 233.016  # experimental H2O atomization energy

_ae_pbe_Ha = 2 * _E_ref["H"]["E_pbe_total"] + _E_ref["O"]["E_pbe_total"] - _E_ref["H2O"]["E_pbe_total"]
PBE_AE_err_kcalmol = abs(_ae_pbe_Ha * 627.509 - _AE_expt_kcalmol)

_ae_ccsd_Ha = 2 * _E_ref["H"]["E_ccsd_total"] + _E_ref["O"]["E_ccsd_total"] - _E_ref["H2O"]["E_ccsd_total"]
CCSD_AE_err_kcalmol = abs(_ae_ccsd_Ha * 627.509 - _AE_expt_kcalmol)

fig, axes = plt.subplots(1, len(LOSS_NAMES), figsize=(6 * len(LOSS_NAMES), 7), squeeze=False)
for col_idx, loss_name in enumerate(LOSS_NAMES):
    ax = axes[0, col_idx]
    n_archs = len(ARCH_NAMES)
    n_solvers = len(SOLVER_LABELS)
    x_positions = np.arange(n_archs)
    bar_width = 0.8 / max(n_solvers, 1)

    for s_idx, solver_label in enumerate(SOLVER_LABELS):
        heights = []
        for arch_name in ARCH_NAMES:
            try:
                val = df.loc[(arch_name, loss_name, solver_label), "AE_error_kcalmol_mean"]
                heights.append(abs(val) if not np.isnan(val) else np.nan)
            except KeyError:
                heights.append(np.nan)
        offset = (s_idx - (n_solvers - 1) / 2) * bar_width
        ax.bar(x_positions + offset, heights, width=bar_width,
               color=solver_colors[solver_label], label=solver_label)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(ARCH_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("|AE error| (kcal/mol)")
    ax.set_yscale("log")
    ax.set_title(f"Loss: {loss_name}", fontsize=11)
    ax.grid(True, which="both", axis="y", ls=":", alpha=0.4)

    ax.axhline(PBE_AE_err_kcalmol, linestyle=":", color="r", linewidth=1.5,
               label=f"PBE ({PBE_AE_err_kcalmol:.2f} kcal/mol)")
    ax.axhline(CCSD_AE_err_kcalmol, linestyle=":", color="b", linewidth=1.5,
               label=f"CCSD ({CCSD_AE_err_kcalmol:.2f} kcal/mol)")
    ax.axhline(1.0, linestyle="--", color="k", alpha=0.7,
               label="Chemical accuracy (1 kcal/mol)")

axes[0, -1].legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize="small",
    title="solver / reference",
)

fig.suptitle(
    "H2O atomization-energy error by architecture and solver config\n"
    "(one subplot per loss family, grouped bars = solver configs)",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
fig.savefig(f"{CHECKPOINT_BASE}/figures/scf_comparison_ae.png", dpi=150, bbox_inches="tight")
plt.show()


### Figure: Density Matrix Residuals -- SCF Comparison

For loss B (energy + DM), this plot compares the density-matrix residuals
(NN - HF target) across the 3 solver configurations for the best-performing
architecture. A 1x3 panel with shared colorbar shows how self-consistency
affects the learned density matrix.


In [ ]:
_loss_b = "B_atomization_plus_dm"
if _loss_b not in LOSS_NAMES:
    print("[Cell 29] loss B not in config -- skipping DM heatmaps")
else:
    # Find best arch for loss B across all solvers
    _sub = df.xs(_loss_b, level="loss")["AE_error_kcalmol_mean"].abs()
    _best_arch, _best_solver = _sub.idxmin()

    dm_hf = mol_data_list[2]["dm_target"]  # H2O is index 2

    fig, axes = plt.subplots(1, len(SOLVER_LABELS), figsize=(5 * len(SOLVER_LABELS), 4.5),
                             squeeze=False)
    _vmax = 0
    _dm_panels = []
    for solver_label in SOLVER_LABELS:
        ckpt = f"{CHECKPOINT_BASE}/train/{_best_arch}/{_loss_b}/{solver_label}/model.eqx"
        if not os.path.isfile(ckpt):
            _dm_panels.append(None)
            continue
        _arch_config = alec.get_architecture(_best_arch)
        _model = eqx.tree_deserialise_leaves(ckpt, alec.AlecGGAModel.from_arch(_arch_config))
        _dm_nn = alec.oneshot_dm_prediction_fast(_model, mol_data_list[2])
        _delta = _dm_nn - dm_hf
        _dm_panels.append(_delta)
        _vmax = max(_vmax, float(jnp.abs(_delta).max()))

    for i, (solver_label, delta) in enumerate(zip(SOLVER_LABELS, _dm_panels)):
        ax = axes[0, i]
        if delta is None:
            ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center")
        else:
            _rmse = float(jnp.sqrt(jnp.mean(delta ** 2)))
            im = ax.imshow(np.asarray(delta), cmap="RdBu_r", vmin=-_vmax, vmax=_vmax)
            ax.set_title(f"{solver_label}\nFrob RMSE={_rmse:.4e}", fontsize=10)
            ax.set_xlabel("AO index j")
            ax.set_ylabel("AO index i")
            fig.colorbar(im, ax=ax, shrink=0.8)

    fig.suptitle(
        f"H2O DM residuals (NN - HF) for loss B, arch={_best_arch}\n"
        f"(shared colorscale, one panel per solver config)",
        fontsize=12,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
    fig.savefig(f"{CHECKPOINT_BASE}/figures/dm_heatmaps_scf.png", dpi=150, bbox_inches="tight")
    plt.show()


### Figure: Grid Density Residuals -- SCF Comparison

For loss C (energy + grid density), this plot overlays histograms of the
grid-weighted density residual (delta-rho) across the 3 solver configurations
for the best-performing architecture. Tighter distributions indicate better
density prediction.


In [ ]:
_loss_c = "C_atomization_plus_grid"
if _loss_c not in LOSS_NAMES:
    print("[Cell 31] loss C not in config -- skipping density histograms")
else:
    _sub = df.xs(_loss_c, level="loss")["AE_error_kcalmol_mean"].abs()
    _best_arch, _best_solver = _sub.idxmin()

    rho_ref = mol_data_list[2]["rho_ref_grid"]
    weights = mol_data_list[2]["grid_weights"]
    _bins = np.linspace(-0.15, 0.15, 81)

    # PBE baseline density residuals
    _rho_pbe = mol_data_list[2]["rho_grid"]
    _delta_pbe = np.asarray(_rho_pbe - rho_ref)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(_delta_pbe, bins=_bins, alpha=0.3, color="red", edgecolor="red",
            linewidth=0.5, label="PBE baseline",
            weights=np.asarray(weights), density=True)

    for solver_label in SOLVER_LABELS:
        ckpt = f"{CHECKPOINT_BASE}/train/{_best_arch}/{_loss_c}/{solver_label}/model.eqx"
        if not os.path.isfile(ckpt):
            continue
        _arch_config = alec.get_architecture(_best_arch)
        _model = eqx.tree_deserialise_leaves(ckpt, alec.AlecGGAModel.from_arch(_arch_config))
        _rho_nn = alec.oneshot_grid_density(_model, mol_data_list[2])
        _delta = np.asarray(_rho_nn - rho_ref)
        ax.hist(_delta, bins=_bins, alpha=0.4, color=solver_colors[solver_label],
                edgecolor=solver_colors[solver_label], linewidth=0.5,
                label=solver_label, weights=np.asarray(weights), density=True)

    ax.set_xlabel(r"$\rho_{\mathrm{NN}} - \rho_{\mathrm{HF}}$  (grid-weighted residual)")
    ax.set_ylabel("probability density (log scale)")
    ax.set_yscale("log")
    ax.set_title(
        f"Grid density residuals vs HF reference for loss C, arch={_best_arch}\n"
        f"(PBE baseline in red, NN models by solver config)",
    )
    ax.legend(title="model / baseline", fontsize="small")
    ax.grid(True, which="both", axis="y", ls=":", alpha=0.4)
    fig.tight_layout()
    os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
    fig.savefig(f"{CHECKPOINT_BASE}/figures/grid_density_scf.png", dpi=150, bbox_inches="tight")
    plt.show()


### Figure: SCF Convergence Diagnostic

This plot runs the SCF solver with increased `max_cycles=10` on a trained
model (deep_combined, loss A) and plots |E(n) - E(n-1)| vs cycle number for
both FIXED_J and FULL modes. This shows the convergence rate under
self-consistency -- how quickly the energy settles.

Note: this is evaluation-only (no training through the loop). The trained
model was trained with the solver config in its checkpoint path; here we
just observe how it behaves under extended iteration.


In [ ]:
from xcquinox.alec.solver import run_scf

_diag_arch = "deep_combined" if "deep_combined" in ARCH_NAMES else ARCH_NAMES[0]
_diag_loss = LOSS_NAMES[0]
_diag_solver = SOLVER_LABELS[0]
_ckpt = f"{CHECKPOINT_BASE}/train/{_diag_arch}/{_diag_loss}/{_diag_solver}/model.eqx"

if not os.path.isfile(_ckpt):
    print(f"[Cell 33] checkpoint not found: {_ckpt} -- skipping convergence plot")
else:
    _arch_config = alec.get_architecture(_diag_arch)
    _model = eqx.tree_deserialise_leaves(_ckpt, alec.AlecGGAModel.from_arch(_arch_config))
    _h2o_data = mol_data_list[2]  # H2O

    _diag_configs = {
        "FIXED_J(10)": SolverConfig(
            backend=SolverBackend.MANUAL,
            mode=SolverMode.FIXED_J,
            max_cycles=10,
            conv_tol=1e-10,
        ),
        "FULL(10)": SolverConfig(
            backend=SolverBackend.MANUAL,
            mode=SolverMode.FULL,
            max_cycles=10,
            conv_tol=1e-10,
        ),
    }

    fig, ax = plt.subplots(figsize=(8, 5))
    for label, cfg in _diag_configs.items():
        result = run_scf(cfg, _model, _h2o_data)
        if hasattr(result, "energy_trace") and result.energy_trace is not None:
            _trace = np.array(result.energy_trace)
            _deltas = np.abs(np.diff(_trace))
            ax.semilogy(range(1, len(_deltas) + 1), _deltas, "o-", label=label)

    ax.set_xlabel("SCF cycle")
    ax.set_ylabel("|E(n) - E(n-1)| (Hartree, log)")
    ax.set_title(
        f"SCF convergence diagnostic -- arch={_diag_arch}, loss={_diag_loss}\n"
        f"(eval-only: trained model run through extended SCF cycles)"
    )
    ax.legend(title="mode (max_cycles=10)")
    ax.grid(True, which="both", ls=":", alpha=0.4)
    fig.tight_layout()
    os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
    fig.savefig(f"{CHECKPOINT_BASE}/figures/scf_convergence.png", dpi=150, bbox_inches="tight")
    plt.show()


### Figure: Feature Impact Across Solver Configs

This plot compares the 4 non-attention deep variants (`deep`, `deep_cusp`,
`deep_dm`, `deep_combined`) across the 3 solver configs. One subplot per loss
family (A, B, C). Using non-attention variants only provides a clean comparison
of descriptor impact without attention confounds.


In [ ]:
_feature_archs = ["deep", "deep_cusp", "deep_dm", "deep_combined"]
_feature_archs = [a for a in _feature_archs if a in ARCH_NAMES]

if not _feature_archs:
    print("[Cell 35] no non-attention deep variants in config -- skipping")
else:
    fig, axes = plt.subplots(1, len(LOSS_NAMES), figsize=(6 * len(LOSS_NAMES), 6),
                             squeeze=False)
    n_archs = len(_feature_archs)
    n_solvers = len(SOLVER_LABELS)
    x_positions = np.arange(n_archs)
    bar_width = 0.8 / max(n_solvers, 1)

    for col_idx, loss_name in enumerate(LOSS_NAMES):
        ax = axes[0, col_idx]
        for s_idx, solver_label in enumerate(SOLVER_LABELS):
            heights = []
            for arch_name in _feature_archs:
                try:
                    val = df.loc[(arch_name, loss_name, solver_label), "AE_error_kcalmol_mean"]
                    heights.append(abs(val) if not np.isnan(val) else np.nan)
                except KeyError:
                    heights.append(np.nan)
            offset = (s_idx - (n_solvers - 1) / 2) * bar_width
            ax.bar(x_positions + offset, heights, width=bar_width,
                   color=solver_colors[solver_label], label=solver_label)

        ax.set_xticks(x_positions)
        ax.set_xticklabels(_feature_archs, rotation=30, ha="right", fontsize=9)
        ax.set_ylabel("|AE error| (kcal/mol)")
        ax.set_yscale("log")
        ax.set_title(f"Loss: {loss_name}", fontsize=11)
        ax.grid(True, which="both", axis="y", ls=":", alpha=0.4)
        ax.axhline(PBE_AE_err_kcalmol, linestyle=":", color="r", linewidth=1.5,
                   label=f"PBE ({PBE_AE_err_kcalmol:.2f} kcal/mol)")
        ax.axhline(CCSD_AE_err_kcalmol, linestyle=":", color="b", linewidth=1.5,
                   label=f"CCSD ({CCSD_AE_err_kcalmol:.2f} kcal/mol)")
        ax.axhline(1.0, linestyle="--", color="k", alpha=0.7,
                   label="Chemical accuracy (1 kcal/mol)")

    axes[0, -1].legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize="small",
        title="solver / reference",
    )

    fig.suptitle(
        "Feature impact: non-attention deep variants x solver config\n"
        "(descriptor dimension increases left to right: 2, 4, 5, 7)",
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
    fig.savefig(f"{CHECKPOINT_BASE}/figures/feature_impact_scf.png", dpi=150, bbox_inches="tight")
    plt.show()


## Section 7: New Molecule Extension

This section demonstrates how to test a new molecule across all 72 trained
models. The pattern:

1. Define a `MoleculeSpec` for the new molecule and any new atomic species.
2. Generate PBE/HF/CCSD reference data and metadata sidecars.
3. Build `atom_energies` from PBE sidecars (PBE-consistent anchoring).
4. Sweep all 72 checkpoints with `alec.run_test`, collecting errors.
5. Plot errors grouped by solver config.


In [ ]:
ext_data_dir = f"{CHECKPOINT_BASE}/external_data"

new_mol_spec = alec.MoleculeSpec(
    name="CH4",
    atom="C 0 0 0; H 0.63 0.63 0.63; H -0.63 -0.63 0.63; H -0.63 0.63 -0.63; H 0.63 -0.63 -0.63",
    basis=BASIS,
    charge=0,
    spin=0,
    atom_composition=(("C", 1), ("H", 4)),
    grid_level=GRID_LEVEL,
    external_data_path=f"{ext_data_dir}/CH4.npz",
)
new_atom_specs = [("C", "C 0 0 0", 2)]

_atom_names = {s[0] for s in new_atom_specs}
_entities = [(new_mol_spec.name, new_mol_spec.atom, new_mol_spec.spin)] + new_atom_specs
os.makedirs(ext_data_dir, exist_ok=True)

for _name, _atom, _spin in _entities:
    _npz_path = f"{ext_data_dir}/{_name}.npz"
    _meta_path = f"{ext_data_dir}/{_name}_metadata.json"
    if os.path.isfile(_npz_path) and os.path.isfile(_meta_path):
        print(f"Using cached {_name} reference data")
        continue

    _mol = gto.M(atom=_atom, basis=BASIS, charge=0, spin=_spin, verbose=0)

    _mf_pbe = dft.UKS(_mol) if _spin else dft.RKS(_mol)
    _mf_pbe.xc = "pbe"
    _mf_pbe.grids.level = GRID_LEVEL
    _mf_pbe.kernel()
    _E_pbe_total = float(_mf_pbe.e_tot)

    _mf_hf = scf.UHF(_mol) if _spin else scf.RHF(_mol)
    _mf_hf.kernel()
    _E_hf_total = float(_mf_hf.e_tot)

    _mycc = cc.UCCSD(_mf_hf) if _spin else cc.CCSD(_mf_hf)
    _mycc.kernel()
    _E_ccsd_total = float(_mf_hf.e_tot + _mycc.e_corr)

    _sidecar = {
        "E_hf_total": _E_hf_total,
        "E_ccsd_total": _E_ccsd_total,
        "E_pbe_total": _E_pbe_total,
        "E_lit_Ha": None,
    }

    _is_atom = _name in _atom_names
    if _is_atom:
        np.savez(_npz_path, E_ref_literature=_E_hf_total)
    else:
        _dm_hf = _mf_hf.make_rdm1()
        _dm_hf_total = _dm_hf[0] + _dm_hf[1] if _dm_hf.ndim == 3 else _dm_hf
        _coords = _mf_pbe.grids.coords
        _weights = _mf_pbe.grids.weights
        _ao = _mf_pbe._numint.eval_ao(_mol, _coords, deriv=0)
        _rho_hf = np.einsum("ij,gi,gj->g", _dm_hf_total, _ao, _ao)

        _dm_pbe = _mf_pbe.make_rdm1()
        _dm_pbe_total = _dm_pbe[0] + _dm_pbe[1] if _dm_pbe.ndim == 3 else _dm_pbe
        _rho_pbe = np.einsum("ij,gi,gj->g", _dm_pbe_total, _ao, _ao)
        _rho_pbe_hf_rmse = float(
            np.sqrt(np.sum(_weights * (_rho_pbe - _rho_hf) ** 2) / np.sum(_weights))
        )

        np.savez(
            _npz_path,
            dm_target=_dm_hf,
            rho_ref_grid=_rho_hf,
            ref_density_method="hf",
            E_ref_literature=float(_mf_hf.e_tot),
        )
        _sidecar["rho_pbe_hf_rmse"] = _rho_pbe_hf_rmse

    with open(_meta_path, "w") as _f:
        json.dump(_sidecar, _f, indent=2)
    print(f"Generated {_name} reference data -> {_npz_path}")

# Build new_atom_energies from PBE sidecars
new_atom_energies = {**atom_energies}
for _name, _atom, _spin in new_atom_specs:
    with open(f"{ext_data_dir}/{_name}_metadata.json") as _f:
        new_atom_energies[_name] = json.load(_f)["E_pbe_total"]
print(f"new_atom_energies (PBE-consistent): {new_atom_energies}")


### Figure: New Molecule Transfer -- SCF Comparison

The next cell sweeps all 72 trained checkpoints on the new molecule (CH4) and
renders a 1x3 panel comparison: AE error, E error, and density RMSE. Each
panel groups bars by solver config and colors by solver, showing whether SCF
self-consistency during training transfers to improved predictions on unseen
molecules.


In [ ]:
_mol_name = new_mol_spec.name
_sweep_rows = []
for _arch in ARCH_NAMES:
    for _loss in LOSS_NAMES:
        for _solver in SOLVER_LABELS:
            _ckpt = f"{CHECKPOINT_BASE}/train/{_arch}/{_loss}/{_solver}/model.eqx"
            if not os.path.isfile(_ckpt):
                continue
            _out_dir = f"{CHECKPOINT_BASE}/test_new/{_mol_name}/{_arch}/{_loss}/{_solver}"
            _spec = alec.TestSpec.from_dicts(
                arch=alec.get_architecture(_arch),
                model_checkpoint=_ckpt,
                molecules=(new_mol_spec,),
                metrics=("total_energy", "atomization_energy", "density_rmse"),
                metric_kwargs={"atomization_energy": {"reference_ae_kcalmol": {_mol_name: 420.0}}},
                atom_energies=new_atom_energies,
                output_dir=_out_dir,
                solver_config=SCF_CONFIGS[_solver],
            )
            _res = alec.run_test(_spec)
            _pm = _res["per_molecule"][0]
            _sweep_rows.append({
                "arch": _arch,
                "loss": _loss,
                "solver": _solver,
                "AE_error_kcalmol": float(abs(_pm.get("AE_error_kcalmol", float("nan")))),
                "E_error_kcalmol": float(abs(_pm.get("E_error_kcalmol", float("nan")))),
                "density_rmse": float(_pm.get("density_rmse", float("nan"))),
            })

if not _sweep_rows:
    print(f"[Cell 39] no checkpoints found -- skipping plot")
else:
    _sweep_df = pd.DataFrame(_sweep_rows)

    # PBE/CCSD reference values for the new molecule
    _new_meta = {}
    for _nm in list(new_atom_energies.keys()) + [_mol_name]:
        _mp = f"{ext_data_dir}/{_nm}_metadata.json"
        if os.path.isfile(_mp):
            with open(_mp) as _f:
                _new_meta[_nm] = json.load(_f)

    _new_pbe_ae = None
    _new_ccsd_ae = None
    _new_pbe_E_err = None
    _new_ccsd_E_err = None
    _ae_ref = 420.0  # experimental AE kcal/mol for CH4
    if _mol_name in _new_meta:
        _comp = dict(new_mol_spec.atom_composition)
        _pbe_atoms = sum(
            _new_meta[Z]["E_pbe_total"] * cnt
            for Z, cnt in _comp.items()
            if Z in _new_meta
        )
        _new_pbe_ae = abs((_pbe_atoms - _new_meta[_mol_name]["E_pbe_total"]) * 627.509 - _ae_ref)
        _ccsd_atoms = sum(
            _new_meta[Z]["E_ccsd_total"] * cnt
            for Z, cnt in _comp.items()
            if Z in _new_meta
        )
        _new_ccsd_ae = abs((_ccsd_atoms - _new_meta[_mol_name]["E_ccsd_total"]) * 627.509 - _ae_ref)
        _hf_total = _new_meta[_mol_name].get("E_hf_total")
        if _hf_total is not None:
            _new_pbe_E_err = abs((_new_meta[_mol_name]["E_pbe_total"] - _hf_total) * 627.509)
            _new_ccsd_E_err = abs((_new_meta[_mol_name]["E_ccsd_total"] - _hf_total) * 627.509)

    # Build sensible x-axis labels from arch/loss
    _unique_combos = _sweep_df[_sweep_df["solver"] == SOLVER_LABELS[0]][["arch", "loss"]].values
    _x_labels = [f"{a}\n{l}" for a, l in _unique_combos]

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    _metrics = [
        ("AE_error_kcalmol", f"|AE error| (kcal/mol)", f"{_mol_name} AE error vs experiment"),
        ("E_error_kcalmol", f"|E error| vs HF (kcal/mol)", f"{_mol_name} total energy error vs HF ref"),
        ("density_rmse", "density RMSE vs HF", f"{_mol_name} density RMSE vs HF ref"),
    ]

    for ax, (col, ylabel, title) in zip(axes, _metrics):
        n_combos = len(_unique_combos)
        for s_idx, solver_label in enumerate(SOLVER_LABELS):
            _sub = _sweep_df[_sweep_df["solver"] == solver_label]
            _x = np.arange(len(_sub))
            ax.bar(_x + s_idx * 0.25, _sub[col].values, width=0.25,
                   color=solver_colors[solver_label], label=solver_label, alpha=0.8)
        ax.set_xticks(np.arange(n_combos) + 0.25)
        ax.set_xticklabels(_x_labels, rotation=60, ha="right", fontsize=6)
        ax.set_ylabel(ylabel)
        ax.set_yscale("log")
        ax.set_title(title, fontsize=10)
        ax.grid(True, which="both", axis="y", ls=":", alpha=0.4)

    # Add reference lines to AE error subplot
    if _new_pbe_ae is not None:
        axes[0].axhline(_new_pbe_ae, linestyle=":", color="r", linewidth=1.5,
                        label=f"PBE ({_new_pbe_ae:.2f})")
    if _new_ccsd_ae is not None:
        axes[0].axhline(_new_ccsd_ae, linestyle=":", color="b", linewidth=1.5,
                        label=f"CCSD ({_new_ccsd_ae:.2f})")
    axes[0].axhline(1.0, linestyle="--", color="k", alpha=0.7,
                    label="Chemical accuracy (1 kcal/mol)")

    # Add reference lines to E error subplot
    if _new_pbe_E_err is not None:
        axes[1].axhline(_new_pbe_E_err, linestyle=":", color="r", linewidth=1.5,
                        label=f"PBE ({_new_pbe_E_err:.1f})")
    if _new_ccsd_E_err is not None:
        axes[1].axhline(_new_ccsd_E_err, linestyle=":", color="b", linewidth=1.5,
                        label=f"CCSD ({_new_ccsd_E_err:.1f})")

    for ax in axes:
        ax.legend(fontsize=7, title="solver / reference", title_fontsize=7)

    fig.suptitle(
        f"Transfer evaluation -- {_mol_name} across all 72 checkpoints\n"
        f"(x-axis = architecture / loss, bars grouped by solver config)",
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    os.makedirs(f"{CHECKPOINT_BASE}/figures", exist_ok=True)
    fig.savefig(f"{CHECKPOINT_BASE}/figures/{_mol_name}_scf_comparison.png",
                dpi=150, bbox_inches="tight")
    plt.show()
